In [107]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [108]:
!pip install faiss-cpu -q

In [109]:
import pandas as pd
import numpy as np
import faiss
import pickle

In [110]:
master_df = pd.read_parquet(
    "/content/drive/MyDrive/PrismHire/master_candidates.parquet"
)

candidate_genome = pd.read_parquet(
    "/content/drive/MyDrive/PrismHire/models/candidate_genome.parquet"
)

master_df = master_df.merge(
    candidate_genome,
    on="candidate_id",
    how="left"
)

master_df.shape

(100000, 40)

In [111]:
candidate_embeddings = np.load(
    "/content/drive/MyDrive/PrismHire/models/candidate_embeddings.npy"
)

candidate_embeddings.shape

(100000, 384)

In [112]:
index = faiss.read_index(
    "/content/drive/MyDrive/PrismHire/models/faiss_index.bin"
)

index.ntotal

100000

In [113]:
master_df = pd.read_parquet(
    "/content/drive/MyDrive/PrismHire/master_candidates.parquet"
)

genome_df = pd.read_parquet(
    "/content/drive/MyDrive/PrismHire/models/candidate_genome.parquet"
)

embeddings = np.load(
    "/content/drive/MyDrive/PrismHire/models/candidate_embeddings.npy"
)

index = faiss.read_index(
    "/content/drive/MyDrive/PrismHire/models/faiss_index.bin"
)

In [114]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [115]:
job_description = """
Senior Backend Engineer

Requirements:
5+ years experience
Python
REST APIs
Microservices
AWS
Docker
"""

In [116]:
job_requirements = {
    'skills': [
        'Python',
        'AWS',
        'Docker',
        'REST APIs',
        'Microservices'
    ],
    'seniority': 'senior',
    'leadership': 0,
    'communication': 1,
    'ownership': 1,
    'startup_adaptability': 1,
    'technical_depth': 0.5
}

In [117]:
job_embedding = model.encode(
    [job_description],
    normalize_embeddings=True
)

In [118]:
distances, indices = index.search(
    job_embedding.astype("float32"),
    1000
)

In [119]:
candidate_pool = master_df.iloc[
    indices[0]
].copy()

candidate_pool.reset_index(
    drop=True,
    inplace=True
)

In [120]:
candidate_pool.shape

(1000, 35)

In [121]:
candidate_pool = candidate_pool.merge(
    genome_df,
    on="candidate_id"
)

AGENT 1 Technical Recruiter Agent

In [122]:
def technical_agent(row, job):

    score = 0

    if row["years_experience"] >= 5:
        score += 0.25

    score += (
        row["technical_dna"] * 0.50
    )

    score += (
        row["avg_assessment_score"] / 100
    ) * 0.25

    return min(score, 1)

AGENT 2 CAREER AGENT

In [123]:
def career_agent(row):

    score = 0

    score += (
        row["career_dna"] * 0.40
    )

    score += (
        min(
            row["avg_job_duration"],
            48
        ) / 48
    ) * 0.30

    score += (
        min(
            row["industry_exposure"],
            5
        ) / 5
    ) * 0.30

    return min(score, 1)

AGENT 3 LEADERSHIP AGENT

In [124]:
def leadership_agent(row):

    score = 0

    score += (
        row["leadership_dna"] * 0.60
    )

    score += (
        min(
            row["leadership_experience"],
            5
        ) / 5
    ) * 0.40

    return min(score, 1)

AGENT 4 RECRUITABILITY AGENT

In [125]:
def recruitability_agent(row):

    score = 0

    score += (
        row["recruitability_dna"] * 0.40
    )

    score += (
        row["recruiter_response_rate"]
    ) * 0.20

    score += (
        row["interview_completion_rate"]
    ) * 0.20

    score += (
        row["open_to_work"]
    ) * 0.20

    return min(score, 1)

AGENT 5 MARKET AGENT

In [126]:
def market_agent(row):

    score = 0

    score += (
        row["market_dna"] * 0.50
    )

    score += (
        min(
            row["saved_by_recruiters"],
            5
        ) / 5
    ) * 0.25

    score += (
        min(
            row["profile_views_30d"],
            50
        ) / 50
    ) * 0.25

    return min(score, 1)

RUN ALL AGENTS

In [127]:
candidate_pool[
    "technical_score"
] = candidate_pool.apply(
    lambda x:
    technical_agent(
        x,
        job_requirements
    ),
    axis=1
)

candidate_pool[
    "career_score"
] = candidate_pool.apply(
    career_agent,
    axis=1
)

candidate_pool[
    "leadership_score"
] = candidate_pool.apply(
    leadership_agent,
    axis=1
)

candidate_pool[
    "recruitability_score"
] = candidate_pool.apply(
    recruitability_agent,
    axis=1
)

candidate_pool[
    "market_score"
] = candidate_pool.apply(
    market_agent,
    axis=1
)

In [128]:
candidate_pool[
    "final_score"
] = (

    candidate_pool[
        "technical_score"
    ] * 0.40

    +

    candidate_pool[
        "career_score"
    ] * 0.20

    +

    candidate_pool[
        "recruitability_score"
    ] * 0.20

    +

    candidate_pool[
        "market_score"
    ] * 0.10

    +

    candidate_pool[
        "leadership_score"
    ] * 0.10
)

In [129]:
final_ranking = (
    candidate_pool
    .sort_values(
        "final_score",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

final_ranking[
    [
        "candidate_id",
        "current_title",
        "years_experience",
        "final_score"
    ]
].head(20)

,candidate_id,current_title,years_experience,final_score
0,CAND_0000110,Software Engineer,9.8,0.602578
1,CAND_0060487,QA Engineer,9.8,0.585584
2,CAND_0036959,Cloud Engineer,10.0,0.583610
3,CAND_0081746,Frontend Engineer,9.7,0.570004
4,CAND_0074198,Java Developer,8.4,0.566894
5,CAND_0019445,Frontend Engineer,5.2,0.565122
6,CAND_0037058,Cloud Engineer,8.6,0.564160
7,CAND_0034856,Mobile Developer,8.1,0.553512
8,CAND_0060901,Mobile Developer,8.8,0.553082
9,CAND_0011130,Frontend Engineer,7.5,0.551868


In [130]:
import pickle

with open(
    "/content/drive/MyDrive/PrismHire/models/skill_set.pkl",
    "rb"
) as f:
    skill_set = pickle.load(f)

In [131]:
def skill_match_score(
        candidate_id,
        required_skills
):

    candidate_skills = skill_set.get(
        candidate_id,
        set()
    )

    matches = 0

    for skill in required_skills:

        if skill.lower() in candidate_skills:
            matches += 1

    return matches / len(required_skills)

In [132]:
skill_match_score(
    "CAND_0000001",
    [
        "AWS",
        "Flask",
        "NLP"
    ]
)

1.0

In [133]:
skill_match_score(
    "CAND_0000001",
    [
        "Python",
        "Docker",
        "SQL"
    ]
)

0.0

In [134]:
backend_titles = [

    "backend engineer",
    "software engineer",
    "senior software engineer",
    "full stack developer",
    "cloud engineer",
    "data engineer"
]

In [135]:
def title_match_score(title):

    title = title.lower()

    for t in backend_titles:
        if t in title:
            return 1

    return 0

In [136]:
title_match_score(
    "Cloud Engineer"
)

1

In [137]:
title_match_score(
    "Frontend Engineer"
)

0

In [138]:
def technical_agent(
        row,
        job
):

    experience_score = min(
        row["years_experience"] / 10,
        1
    )

    skill_score = skill_match_score(
        row["candidate_id"],
        job["skills"]
    )

    title_score = title_match_score(
        row["current_title"]
    )

    assessment_score = (
        row["avg_assessment_score"]
        / 100
    )

    score = (

        experience_score * 0.20
        +
        skill_score * 0.40
        +
        title_score * 0.20
        +
        assessment_score * 0.10
        +
        row["technical_dna"] * 0.10

    )

    return score

In [139]:
candidate_pool[
    "technical_score"
] = candidate_pool.apply(
    lambda x:
    technical_agent(
        x,
        job_requirements
    ),
    axis=1
)

In [140]:
candidate_pool[
    "final_score"
] = (

    candidate_pool["technical_score"] * 0.40
    +
    candidate_pool["career_score"] * 0.20
    +
    candidate_pool["recruitability_score"] * 0.20
    +
    candidate_pool["market_score"] * 0.10
    +
    candidate_pool["leadership_score"] * 0.10
)

In [141]:
final_ranking = (
    candidate_pool
    .sort_values(
        "final_score",
        ascending=False
    )
    .reset_index(drop=True)
)

final_ranking[
    [
        "candidate_id",
        "current_title",
        "years_experience",
        "technical_score",
        "final_score"
    ]
].head(20)

,candidate_id,current_title,years_experience,technical_score,final_score
0,CAND_0036959,Cloud Engineer,10.0,0.661959,0.609575
1,CAND_0000110,Software Engineer,9.8,0.674716,0.600233
2,CAND_0033851,Full Stack Developer,9.1,0.716190,0.594632
3,CAND_0046701,Software Engineer,8.0,0.705884,0.571056
4,CAND_0026597,Cloud Engineer,4.3,0.623946,0.565702
5,CAND_0080041,Cloud Engineer,6.3,0.651755,0.563572
6,CAND_0075634,Software Engineer,8.7,0.733566,0.558435
7,CAND_0081516,Cloud Engineer,9.1,0.575072,0.547520
8,CAND_0001023,Software Engineer,6.2,0.656667,0.543907
9,CAND_0030723,Cloud Engineer,8.9,0.547392,0.541609


ROLE SIMILARITY AGENT

In [142]:
master_df.shape

(100000, 35)

In [143]:
genome_df.shape

(100000, 6)

In [144]:
indices

array([[21574, 68631, 50623, 51499, 30215, 15825, 88423,  1398, 60362,
        71958, 61694, 46205, 29309, 86563, 54731, 23521, 90542, 16087,
        29691, 68120,  1290, 87286, 94654, 35962, 34855, 96264, 22129,
        63685, 45143, 90691, 55318, 25851, 28121, 84031, 12799,  5653,
        25867, 89082, 46700,  8419, 73439, 45578, 32877, 94979,  7040,
        15496, 83734, 68255, 87478, 24209, 36958, 25748, 65312, 80511,
        21618, 32982,  7230, 31190, 42890, 10475, 58926, 33812, 48986,
         3112, 11763, 71029, 25314, 24077, 14735, 44135, 97613, 63782,
        54845, 82929, 65207, 28912, 58390, 41656, 56709, 86655, 99341,
        79863, 79226, 87094, 11129, 28620, 63204, 61024, 10788,  7361,
        90020, 99413,  2640, 93935, 23511, 19819, 78131,  1521, 42300,
        26946, 83479, 12192,  9957, 40751, 56152, 36075, 74922, 76178,
        92430,  9895, 74051, 66072, 28773, 87959, 57037, 96344, 48898,
         9014, 71853, 34916, 96740, 93483, 96025, 79086, 76941, 85666,
      

In [145]:
results = master_df.iloc[
    indices[0]
].copy()

In [146]:
print("master_df:", "master_df" in globals())
print("genome_df:", "genome_df" in globals())
print("index:", "index" in globals())
print("model:", "model" in globals())
print("job_embedding:", "job_embedding" in globals())
print("scores:", "scores" in globals())
print("indices:", "indices" in globals())

master_df: True
genome_df: True
index: True
model: True
job_embedding: True
scores: True
indices: True


In [147]:
scores, indices = index.search(
    job_embedding,
    500
)

In [148]:
scores.shape

(1, 500)

In [149]:
indices.shape

(1, 500)

In [150]:
results = master_df.iloc[
    indices[0]
].copy()

results["semantic_score"] = scores[0]

In [151]:
results.head()

,candidate_id,years_experience,current_title,current_industry,current_company_size,country,num_skills,avg_skill_proficiency,avg_skill_endorsements,avg_skill_duration,...,connection_count,endorsements_received,notice_period_days,willing_to_relocate,preferred_work_mode,verified_email,verified_phone,linkedin_connected,avg_assessment_score,semantic_score
21574,CAND_0021575,7.1,Cloud Engineer,Fintech,1001-5000,India,9,1.555556,7.222222,19.777778,...,274,2,120,1,2,0,1,0,0.0,0.641164
68631,CAND_0068632,7.6,Cloud Engineer,IT Services,10001+,India,8,1.125000,8.125000,12.125000,...,109,14,30,0,4,0,1,0,0.0,0.628155
50623,CAND_0050624,4.0,Cloud Engineer,E-commerce,10001+,India,8,1.750000,15.750000,19.875000,...,98,48,90,1,3,1,0,0,0.0,0.626739
51499,CAND_0051500,5.6,Cloud Engineer,IT Services,10001+,India,9,1.666667,9.000000,19.111111,...,303,41,30,0,3,1,1,1,0.0,0.624475
30215,CAND_0030216,6.9,Cloud Engineer,IT Services,10001+,Singapore,11,1.363636,7.454545,15.636364,...,194,19,120,1,2,1,1,0,0.0,0.622009


In [152]:
results = results.merge(
    genome_df,
    on="candidate_id",
    how="left"
)

In [153]:
results["technical_score"] = results["technical_dna"]
results["career_score"] = results["career_dna"]
results["market_score"] = results["market_dna"]
results["leadership_score"] = results["leadership_dna"]
results["recruitability_score"] = results["recruitability_dna"]

In [154]:
backend_titles = {
    "backend engineer",
    "software engineer",
    "senior software engineer",
    "full stack developer",
    "cloud engineer",
    "java developer",
    "devops engineer"
}


In [155]:
backend_roles = [
    "backend engineer",
    "software engineer",
    "full stack developer",
    "cloud engineer",
    "java developer",
    "backend developer"
]

results = results[
    results["current_title"]
    .str.lower()
    .isin(backend_roles)
].copy()

In [156]:
def role_similarity(title):

    title = str(title).lower()

    score_map = {
        "backend engineer": 1.0,
        "software engineer": 0.85,
        "full stack developer": 0.80,
        "cloud engineer": 0.75,
        "java developer": 0.75,
        "devops engineer": 0.70,
        "mobile developer": 0.50,
        "frontend engineer": 0.40,
        "qa engineer": 0.20
    }

    for role, score in score_map.items():
        if role in title:
            return score

    return 0.10

In [157]:
results["role_score"] = (
    results["current_title"]
    .apply(role_similarity)
)

In [158]:
job_description

'\nSenior Backend Engineer\n\nRequirements:\n5+ years experience\nPython\nREST APIs\nMicroservices\nAWS\nDocker\n'

In [159]:
job_profile = {
    "skills": [
        "Python",
        "AWS",
        "Docker",
        "REST APIs",
        "Microservices"
    ],
    "seniority": "senior",
    "leadership": 0,
    "communication": 1,
    "ownership": 1,
    "startup_adaptability": 1,
    "technical_depth": 0.5
}

In [160]:
required_skills = {
    s.lower()
    for s in job_profile["skills"]
}

In [161]:
def compute_skill_overlap(skill_set):

    candidate_skills = {
        s.lower()
        for s in skill_set
    }

    overlap = len(
        required_skills &
        candidate_skills
    )

    return overlap / len(required_skills)

In [162]:
results.columns.tolist()

['candidate_id',
 'years_experience',
 'current_title',
 'current_industry',
 'current_company_size',
 'country',
 'num_skills',
 'avg_skill_proficiency',
 'avg_skill_endorsements',
 'avg_skill_duration',
 'num_jobs',
 'avg_job_duration',
 'max_job_duration',
 'industry_exposure',
 'company_scale_exposure',
 'leadership_experience',
 'open_to_work',
 'profile_completeness',
 'recruiter_response_rate',
 'avg_response_time_hours',
 'profile_views_30d',
 'search_appearances',
 'saved_by_recruiters',
 'interview_completion_rate',
 'offer_acceptance_rate',
 'github_activity_score',
 'connection_count',
 'endorsements_received',
 'notice_period_days',
 'willing_to_relocate',
 'preferred_work_mode',
 'verified_email',
 'verified_phone',
 'linkedin_connected',
 'avg_assessment_score',
 'semantic_score',
 'technical_dna',
 'career_dna',
 'leadership_dna',
 'recruitability_dna',
 'market_dna',
 'technical_score',
 'career_score',
 'market_score',
 'leadership_score',
 'recruitability_score',
 'r

In [163]:
required_exp = 5

In [164]:
def experience_score(exp):

    if exp >= required_exp:
        return 1.0

    return exp / required_exp

In [165]:
results["experience_score"] = (
    results["years_experience"]
    .apply(experience_score)
)

In [166]:
results[
    [
        "current_title",
        "years_experience",
        "experience_score"
    ]
].head()

,current_title,years_experience,experience_score
0,Cloud Engineer,7.1,1.0
1,Cloud Engineer,7.6,1.0
2,Cloud Engineer,4.0,0.8
3,Cloud Engineer,5.6,1.0
4,Cloud Engineer,6.9,1.0


In [167]:
results["estimated_skill_score"] = (
      0.40 * results["avg_skill_proficiency"] / 4
    + 0.30 * results["avg_skill_endorsements"] /
      results["avg_skill_endorsements"].max()
    + 0.30 * results["num_skills"] /
      results["num_skills"].max()
)

In [168]:
results["final_score"] = (
      0.25 * results["semantic_score"]
    + 0.20 * results["technical_score"]
    + 0.10 * results["estimated_skill_score"]
    + 0.10 * results["career_score"]
    + 0.10 * results["market_score"]
    + 0.05 * results["recruitability_score"]
    + 0.10 * results["role_score"]
    + 0.10 * results["experience_score"]
)

In [169]:
results = (
    results
    .sort_values(
        "final_score",
        ascending=False
    )
    .reset_index(drop=True)
)

In [170]:
final_results = []

for title in results["current_title"].unique():

    candidate = (
        results[
            results["current_title"] == title
        ]
        .head(1)
    )

    final_results.append(candidate)

diverse_results = (
    pd.concat(final_results)
    .sort_values(
        "final_score",
        ascending=False
    )
)

In [171]:
diverse_results[
    [
        "candidate_id",
        "current_title",
        "years_experience",
        "final_score"
    ]
].head(20)

,candidate_id,current_title,years_experience,final_score
0,CAND_0075634,Software Engineer,8.7,0.561279
1,CAND_0033851,Full Stack Developer,9.1,0.559816
2,CAND_0087095,Cloud Engineer,7.2,0.557272
99,CAND_0075434,Java Developer,7.5,0.493554


In [172]:
def explain_candidate(row):

    reasons = []

    # Experience Agent
    if row["years_experience"] >= 8:
        reasons.append(
            f"✓ {row['years_experience']:.1f} years of senior-level experience"
        )
    elif row["years_experience"] >= 5:
        reasons.append(
            f"✓ {row['years_experience']:.1f} years of relevant experience"
        )
    else:
        reasons.append(
            f"⚠ Only {row['years_experience']:.1f} years of experience"
        )

    # Semantic Agent
    if row["semantic_score"] >= 0.60:
        reasons.append(
            "✓ Strong alignment with job requirements"
        )
    elif row["semantic_score"] >= 0.45:
        reasons.append(
            "✓ Moderate alignment with job requirements"
        )

    # Technical Agent
    if row["technical_score"] >= 0.60:
        reasons.append(
            "✓ Excellent technical skill compatibility"
        )
    elif row["technical_score"] >= 0.35:
        reasons.append(
            "✓ Good technical skill compatibility"
        )

    # Career Agent
    if row["career_score"] >= 0.35:
        reasons.append(
            "✓ Demonstrates strong career progression"
        )

    # Recruitability Agent
    if row["recruitability_score"] >= 0.60:
        reasons.append(
            "✓ Highly recruitable and responsive profile"
        )
    elif row["recruitability_score"] >= 0.35:
        reasons.append(
            "✓ Good recruiter engagement signals"
        )

    # Market Agent
    if row["market_score"] >= 0.25:
        reasons.append(
            "✓ Possesses market-relevant skills"
        )

    # Role Agent
    if row["role_score"] >= 0.80:
        reasons.append(
            "✓ Excellent role fit"
        )
    elif row["role_score"] >= 0.50:
        reasons.append(
            "✓ Good role fit"
        )
    else:
        reasons.append(
            "⚠ Role alignment could be improved"
        )

    # Final Recommendation
    if row["final_score"] >= 0.60:
        recommendation = "Highly Recommended"
    elif row["final_score"] >= 0.45:
        recommendation = "Recommended"
    else:
        recommendation = "Consider with Caution"

    return reasons, recommendation

In [173]:
results["explanations"] = results.apply(explain_candidate, axis=1)

In [174]:
def format_output(row):
    return f"""
Candidate ID: {row['candidate_id']}
Current Role: {row['current_title']}
Final Score: {round(row['final_score'], 4)}

Why this candidate ranked here:
{chr(10).join(row['explanations'])}
"""

In [175]:
def explain_with_weights(row):

    breakdown = []

    breakdown.append(f"Semantic contribution: {round(row['semantic_score'] * 0.25, 3)}")
    breakdown.append(f"Technical contribution: {round(row['technical_score'] * 0.25, 3)}")
    breakdown.append(f"Career contribution: {round(row['career_score'] * 0.15, 3)}")
    breakdown.append(f"Market contribution: {round(row['market_score'] * 0.10, 3)}")
    breakdown.append(f"Recruitability contribution: {round(row['recruitability_score'] * 0.10, 3)}")
    breakdown.append(f"Role contribution: {round(row['role_score'] * 0.05, 3)}")
    breakdown.append(f"Experience contribution: {round(row['experience_score'] * 0.10, 3)}")

    return breakdown

In [176]:
results["score_breakdown"] = results.apply(explain_with_weights, axis=1)

In [177]:
def format_output_v2(row):
    return f"""
Candidate ID: {row['candidate_id']}
Role: {row['current_title']}
Final Score: {round(row['final_score'], 4)}

WHY THIS SCORE:
{chr(10).join(row['explanations'])}

SCORE BREAKDOWN:
{chr(10).join(row['score_breakdown'])}
"""

In [178]:
results[
    [
        "candidate_id",
        "current_title",
        "years_experience",
        "semantic_score",
        "technical_score",
        "career_score",
        "market_score",
        "recruitability_score",
        "role_score",
        "experience_score",
        "final_score"
    ]
].head(20)

,candidate_id,current_title,years_experience,semantic_score,technical_score,career_score,market_score,recruitability_score,role_score,experience_score,final_score
0,CAND_0075634,Software Engineer,8.7,0.592878,0.468657,0.354547,0.172134,0.512191,0.85,1.00,0.561279
1,CAND_0033851,Full Stack Developer,9.1,0.592552,0.417898,0.363453,0.243230,0.596152,0.80,1.00,0.559816
2,CAND_0087095,Cloud Engineer,7.2,0.602638,0.499257,0.257892,0.305284,0.353980,0.75,1.00,0.557272
3,CAND_0036959,Cloud Engineer,10.0,0.607778,0.368593,0.400941,0.235836,0.711688,0.75,1.00,0.552385
4,CAND_0084032,Cloud Engineer,4.3,0.610324,0.523724,0.191884,0.320597,0.493389,0.75,0.86,0.551461
5,CAND_0082175,Cloud Engineer,9.1,0.587432,0.496094,0.329893,0.228808,0.350570,0.75,1.00,0.550883
6,CAND_0054732,Cloud Engineer,8.7,0.613870,0.450331,0.345027,0.270431,0.300374,0.75,1.00,0.550045
7,CAND_0097614,Cloud Engineer,8.6,0.605197,0.441221,0.350326,0.314985,0.212414,0.75,1.00,0.549284
8,CAND_0015497,Cloud Engineer,9.1,0.608192,0.490938,0.299893,0.160278,0.331968,0.75,1.00,0.548733
9,CAND_0030723,Cloud Engineer,8.9,0.590258,0.332423,0.364101,0.296244,0.643402,0.75,1.00,0.546291


NORMALIZATION + CALIBRATION

In [179]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

score_cols = [
    "semantic_score",
    "technical_score",
    "career_score",
    "market_score",
    "recruitability_score",
    "role_score",
    "experience_score"
]

results[score_cols] = scaler.fit_transform(results[score_cols])

In [180]:
final_score = (
    results[score_cols].mean(axis=1)
)

FINAL PIPELINE

In [181]:
final_output = results[
    [
        "candidate_id",
        "current_title",
        "years_experience",
        "semantic_score",
        "technical_score",
        "career_score",
        "market_score",
        "recruitability_score",
        "role_score",
        "experience_score",
        "final_score"
    ]
]

final_output.to_csv("top_candidates_ranked.csv", index=False)

In [182]:
PROJECT_PATH = "/content/drive/MyDrive/PrismHire"

DATA_PATH = f"{PROJECT_PATH}/data"
OUTPUT_PATH = f"{PROJECT_PATH}/outputs"
MODEL_PATH = f"{PROJECT_PATH}/models"
GRAPH_PATH = f"{PROJECT_PATH}/graphs"

In [183]:
final_output.to_csv(f"{OUTPUT_PATH}/top_candidates_ranked.csv", index=False)

In [184]:
results["current_title"].value_counts()

,count
current_title,
Cloud Engineer,293
Software Engineer,42
Full Stack Developer,29
Java Developer,5
